# RAPID travel-time calibration

Fit **one effective Manning's n per original network** to the selected event's observed upstream-to-downstream peak lag. Only `S000_base` is routed. BASED depth anchors, depth–width exponent **0.65**, observed-derived widths and slopes, segmentation, and Muskingum **X = 0.1** remain fixed. No separate-event validation is included.

This notebook reads **Routing windows** in `../reviews/hydrograph_match_review.xlsx`. It saves isolated calibration candidates and leaves production results and workbook contents intact. Run cells from top to bottom. The configuration cell selects the networks and exploratory n range.

**Refinement update:** sections 9–12 extend and refine the initial search, compare the fitted values with published roughness guidance, and retain the original search above for comparison.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RAPID/src').is_dir())
sys.path.insert(0, str(ROOT / 'RAPID/investigations'))
import travel_time_calibration as cal

plt.rcParams.update({'figure.dpi': 115, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.max_rows', 30)

## 1. Reviewed windows and expected savings

The requested bounds are **start-inclusive and end-exclusive, in UTC**. First/last retained forcing samples follow the existing sampling grid. The initial and final plateau discharges can differ.

The Excel schedule was tested at n = 0.035 on 39 selected states across 16 networks. Its tail is an initial plan for calibration: each new n must satisfy settlement again. The worksheet's linked event dates and duration formulas are checked against their source cells; stale cached values stop the read.

| Worksheet content | Cells |
|---|---|
| Requested event start / exclusive end | B7:C22 |
| First / last retained samples and plateau discharges | D7:G22 |
| Spin-up, event, settlement and final buffer | B28:G43 |
| Previous production dates and duration comparison | B49:G64 |

In [ ]:
schedule = cal.read_schedule()
display(schedule[['example','requested_start','requested_end_exclusive','old_days','new_days',
                  'duration_reduction_pct','tested_states']].reset_index(drop=True).round(2))
parent_reduction = 100 * (1 - schedule.new_days.sum()/schedule.old_days.sum())
print(f'One run per base network: {schedule.old_days.sum():.1f} → {schedule.new_days.sum():.1f} '
      f'simulated network-days ({parent_reduction:.1f}% reduction).')
print('This is a duration comparison, not a measured wall-clock speedup. Calibration tails may extend.')

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
x = np.arange(len(schedule))
ax.bar(x-.2, schedule.old_days, .4, label='Previous production')
ax.bar(x+.2, schedule.new_days, .4, label='Reviewed schedule')
ax.set(xticks=x, xticklabels=schedule.example, ylabel='Simulation duration (days)', xlabel='Example')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 2. Objective and search settings

\[
\tau_{obs}=t^{down}_{peak}-t^{up}_{peak},\qquad
\tau_{sim}(n)=t^{out}_{peak}(n)-t^{forcing}_{peak},\qquad
J(n)=|\tau_{sim}(n)-\tau_{obs}|.
\]

Use the **first sampled global maximum** consistently. Search observed peaks within the requested event window; search the modeled outlet over the complete routed response, including its tail. Do not truncate a late outlet peak to the observation window. Forcing resampling may shift its peak relative to the upstream gauge, so the notebook also reports that offset and absolute downstream peak error separately.

The n range below is an **exploratory search bracket**, not a physical confidence interval. A boundary optimum calls for reviewing the bracket and event interpretation. A coarse grid is followed by seven local refinement candidates around an interior optimum. Exact score ties select the value closest to the existing n = 0.035; all near-best candidates remain visible.

For an indicative sampling tolerance, use the sum of upstream and downstream median sampling intervals, or one forcing interval if larger. This is a practical comparison tolerance, not measurement uncertainty or a confidence interval. Tied maxima, data gaps and multiple event features need separate interpretation.

In [ ]:
# Edit these controls, then rerun from this cell.
EXAMPLES = schedule.example.tolist()  # e.g. ['02', '46'] for a smaller run
N_GRID = np.array([.015, .020, .025, .030, .035, .040, .050, .060, .080])
REFINE_POINTS = 7
OUTPUT = ROOT / 'RAPID/outputs/travel_time_calibration'
OUTPUT.mkdir(parents=True, exist_ok=True)

# Exact input/implementation identities protect trial-cache reuse.
# New n bounds can change the shared routing clock and invalidate old candidates.
cases = {}
for ex in EXAMPLES:
    cases[ex] = cal.load_case(ex, schedule, n_bounds=(float(N_GRID.min()), float(N_GRID.max())))
    print(f'{ex}: {int(cases[ex]["frame"]["count"].sum()):,} segments; '
          f'routing dt {cases[ex]["routing_dt_s"]:.4g} s', flush=True)

## 3. Inspect observed timing targets

Exact tied peaks are reported as first/last maxima and a span. That span may include distinct equal peaks; the notes identify disconnected ties. A target whose peak touches the event boundary or whose observed lag is nonpositive is excluded from automatic fitting.

The workbook's **Review** and **Peak timing** tabs retain important interpretation: case 03 has a maximum plateau; cases 05 and 11 have alternative proposed features; case 48 has multiple candidate features. Those alternatives are not silently substituted for global maxima here. Improved first-maximum lag alone does not prove that upstream and downstream maxima represent the same wave, especially where lateral inflows matter.

In [ ]:
targets = pd.DataFrame([{k:v for k,v in c['target'].items() if k != 'series'} for c in cases.values()])
display(targets[['example','observed_lag_h','upstream_peak_utc','downstream_peak_utc',
                 'upstream_tied_span_h','downstream_tied_span_h','target_usable','target_notes']])
targets.to_csv(OUTPUT / 'observed_targets.csv', index=False)
schedule.to_csv(OUTPUT / 'reviewed_schedule.csv', index=False)

## 4. Route the existing n = 0.035 baseline

Each trial recomputes celerity and K from the frozen depth/slope geometry, including the **0.05–5 m/s celerity caps**. One common clock per case covers the full n search bracket and is no larger than its saved production clock; this avoids parameter-dependent timestep changes within the search. The output/forcing cadence stays fixed.

Routing uses the existing spatial evaluation of the native Muskingum recurrence with the production split allocations. Each trial is compared against production sparse operations on a forcing prefix; every segment is checked for finite, nonnegative flow at every native substep. The selected search clock is a numerical choice, not a fitted parameter.

Initialize at equilibrium with the first retained discharge. Save the reviewed 24-hour initial plateau without repeatedly simulating an exactly steady state. Start with the workbook's tail, then extend it if needed until the final outlet is within **0.1% of that trial's peak excess above max(initial, final plateau)** for 24 hours, followed by another 24-hour buffer. Settlement uses the last exceedance, so a temporary early crossing cannot end the run.

In [ ]:
baseline_rows = []
for ex, case in cases.items():
    result, _ = cal.run_trial(case, cal.BASE_N, OUTPUT)
    baseline_rows.append(result)
    print(f'{ex}: observed {result["observed_lag_h"]:.2f} h; '
          f'baseline {result["simulated_lag_h"]:.2f} h', flush=True)
baseline = pd.DataFrame(baseline_rows)
display(baseline[['example','observed_lag_h','simulated_lag_h','signed_error_h',
                  'forcing_minus_observed_upstream_peak_h','tail_extensions','capped_segment_fraction']].round(3))
baseline.to_csv(OUTPUT / 'baseline.csv', index=False)

## 5. Coarse search and local refinement

Trials are cached by workbook, geometry, forcing, observed data, code, n and clock. A failed numerical or settlement check stops that case rather than reporting it as a good timing fit. Results are candidates for this selected event; nothing is promoted to production.

In [ ]:
trials_by_example, best_rows = {}, []
for ex, case in cases.items():
    if not case['target']['target_usable']:
        print(f'{ex}: no automatic fit — {case["target"]["target_notes"]}', flush=True)
        continue
    trials, best = cal.calibrate(case, N_GRID, OUTPUT, refine_points=REFINE_POINTS)
    trials_by_example[ex] = trials
    best_rows.append(best)
    print(f'{ex}: n={best["n_manning"]:.5f}; |lag error|={best["abs_error_h"]:.3f} h; '
          f'boundary={best["best_at_search_boundary"]}', flush=True)

best_candidates = pd.DataFrame(best_rows)
if len(best_candidates):
    display(best_candidates[['example','n_manning','observed_lag_h','simulated_lag_h',
                             'baseline_abs_error_h','abs_error_h','timing_resolution_h',
                             'best_at_search_boundary','near_best_n_values','target_notes']].round(4))
    best_candidates.to_csv(OUTPUT / 'best_candidates.csv', index=False)

In [ ]:
if trials_by_example:
    ncols = 4
    nrows = int(np.ceil(len(trials_by_example)/ncols))
    fig, axes = plt.subplots(nrows,ncols,figsize=(14,3*nrows),squeeze=False)
    for ax, (ex, trial) in zip(axes.flat,trials_by_example.items()):
        ax.plot(trial.n_manning, trial.simulated_lag_h, 'o-', color='#2563eb', ms=3)
        ax.axhline(trial.observed_lag_h.iloc[0], color='#111827', ls='--', label='Observed lag')
        ax.axvline(cal.BASE_N, color='#94a3b8', ls=':', label='Existing n')
        best = best_candidates.loc[best_candidates.example.eq(ex)].iloc[0]
        ax.scatter(best.n_manning,best.simulated_lag_h,color='#dc2626',zorder=5)
        ax.set(title=f'{ex}'+(' (boundary)' if best.best_at_search_boundary else ''),
               xlabel='Manning n',ylabel='Peak lag (h)')
    for ax in list(axes.flat)[len(trials_by_example):]: ax.set_visible(False)
    axes.flat[0].legend(frameon=False,fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT / 'travel_time_search.png',dpi=160)
    plt.show()

## 6. Inspect an event on its UTC clock

Change `PLOT_EXAMPLE` to any fitted example. The lower panel normalizes each series independently to show timing and shape; it does not shift timestamps. Peak lag is the only fitted objective. Amplitude mismatch is visible as context and does not affect the score.

In [ ]:
PLOT_EXAMPLE = '02'
if PLOT_EXAMPLE in trials_by_example:
    case = cases[PLOT_EXAMPLE]
    best = best_candidates.loc[best_candidates.example.eq(PLOT_EXAMPLE)].iloc[0]
    _, before = cal.run_trial(case,cal.BASE_N,OUTPUT)
    _, after = cal.run_trial(case,float(best.n_manning),OUTPUT)
    fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True)
    series = [
        (case['target']['series']['upstream'].time,case['target']['series']['upstream'].discharge_cms,'Observed upstream','#94a3b8'),
        (case['target']['series']['downstream'].time,case['target']['series']['downstream'].discharge_cms,'Observed downstream','#111827'),
        (before.time_utc,before.q_outlet_cms,'Baseline n=0.035','#2563eb'),
        (after.time_utc,after.q_outlet_cms,f'Candidate n={best.n_manning:.5f}','#dc2626'),
    ]
    for times, values, label, color in series:
        values = np.asarray(values)
        axes[0].plot(times,values,label=label,color=color,lw=1.4)
        axes[1].plot(times,(values-values.min())/np.ptp(values),label=label,color=color,lw=1.4)
    axes[0].set(ylabel='Discharge (m³/s)',title=f'Example {PLOT_EXAMPLE}: selected event, no time alignment')
    axes[1].set(ylabel='Independent min–max scale',xlabel='Time (UTC)')
    axes[0].legend(frameon=False,ncol=2)
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    axes[1].set_xlim(case['window'].first_sample-pd.Timedelta(hours=6),
                     max(case['window'].requested_end_exclusive,pd.Timestamp(best.model_peak_utc)+pd.Timedelta(days=1)))
    fig.tight_layout()
    fig.savefig(OUTPUT / f'{PLOT_EXAMPLE}_calibration_hydrograph.png',dpi=160)
    plt.show()

## 7. Check sensitivity to the numerical clock

Rerun each best candidate with half the routing timestep, retaining the same
forcing/output cadence, n and geometry. Report the change in sampled peak lag;
a difference larger than one forcing interval needs numerical review before
interpreting the selected n. These are clock checks on the same event.

In [ ]:
clock_rows = []
for best in best_rows:
    original = cases[best['example']]
    fine = dict(original)
    fine['routing_dt_s'] = original['routing_dt_s']/2
    fine['nsub'] = original['nsub']*2
    fine['signature'] = hashlib.sha256((original['signature']+':half-clock').encode()).hexdigest()
    check, _ = cal.run_trial(fine,best['n_manning'],OUTPUT)
    difference_s = 3600*(check['simulated_lag_h']-best['simulated_lag_h'])
    clock_rows.append(dict(example=best['example'],n_manning=best['n_manning'],
                           routing_dt_s=original['routing_dt_s'],half_dt_s=fine['routing_dt_s'],
                           lag_change_seconds=difference_s,forcing_interval_s=original['forcing_step_s'],
                           within_one_forcing_interval=abs(difference_s)<=original['forcing_step_s']+1e-8))
clock_checks = pd.DataFrame(clock_rows)
display(clock_checks.round(4))
clock_checks.to_csv(OUTPUT / 'clock_checks.csv',index=False)

## 8. Numerical and scope checks

The shortened dates are supported as a starting schedule; settlement can change with n. The cap fraction below helps identify trials whose timing is limited by celerity clipping. An unchanged score over a range of n may also come from sampled peak times or competing maxima.

The suggested next scientific step is to review the fitted candidates and peak correspondence, then freeze one n per case and apply it unchanged to variants. No child networks or separate events are fitted here. Any future variant run must recheck its own timestep and final settlement.

In [ ]:
if trials_by_example:
    all_trials = pd.concat(trials_by_example.values(),ignore_index=True)
    numerical = all_trials.groupby('example').agg(
        trials=('n_manning','size'), minimum_native_flow_cms=('minimum_native_flow_cms','min'),
        max_sparse_difference_cms=('sparse_parity_max_cms','max'),
        max_tail_extensions=('tail_extensions','max'),
        max_capped_segment_fraction=('capped_segment_fraction','max'),
        routing_seconds=('routing_seconds','sum'))
    display(numerical.round(6))
    all_trials.to_csv(OUTPUT / 'all_trials.csv',index=False)
    numerical.to_csv(OUTPUT / 'numerical_checks.csv')
    print('Candidate summaries, hydrographs and per-segment numerical audits:', OUTPUT)

manifest = dict(workbook=str(cal.WORKBOOK),workbook_sha256=schedule.attrs['workbook_sha256'],
                objective='absolute first-sampled-maximum lag error in hours',examples=EXAMPLES,
                n_grid=N_GRID.tolist(),refine_points=REFINE_POINTS,
                fixed=dict(depth_width_exponent=.65,x=.1,celerity_limits_mps=[.05,5.],
                           geometry='stored production base networks',depth='stored BASED anchors'),
                case_signatures={ex:c['signature'] for ex,c in cases.items()},
                production_changed=False,separate_events=False)
(OUTPUT / 'notebook_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Workbook and production inputs remain unchanged.')

Sources: [review workbook](../reviews/hydrograph_match_review.xlsx), [buffer-test evidence](../validation/event_buffer_tests_20261005/README.md), [production hydraulic policy](../HYDRAULIC_PRODUCTION_POLICY.md), and [calibration implementation](../investigations/travel_time_calibration.py). Gauges are loaded through the existing station lookup and processed subdaily observation loader. First-maximum conventions are retained from the review; proposed shoulders and bump references remain exploratory.

## 9. Published Manning n guidance

There is **no universal acceptable n interval** for natural rivers. Bed forms, obstructions, vegetation, water depth and reach geometry matter. These published examples give context:

| Channel or bed condition | Manning n range | Source |
|---|---:|---|
| Straight, unobstructed natural channel | 0.025–0.033 | HEC-RAS, Table 3-1 |
| Winding natural channels with pools, stones or weeds (combined examples) | 0.033–0.060 | HEC-RAS, Table 3-1 |
| Slow, vegetated channel with pools | 0.050–0.080 | HEC-RAS, Table 3-1 |
| Strong vegetation or timber obstruction | 0.070–0.150 | HEC-RAS, Table 3-1 |
| Alluvial plane bed | 0.012–0.022 | HEC-RAS, Table 8-1 |
| Alluvial dunes | 0.020–0.035 | HEC-RAS, Table 8-1 |
| Upper-regime sand plane bed, no vegetation, D50 < 1 mm | 0.010–0.013 (base roughness) | FHWA HEC-20, Table 3.3 |

Sources checked 5 October 2026: [HEC-RAS natural-channel table](https://www.hec.usace.army.mil/confluence/rasdocs/ras1dtechref/latest/basic-data-requirements/geometric-data/energy-loss-coefficients), [HEC-RAS alluvial-bed guidance](https://www.hec.usace.army.mil/confluence/rasdocs/rasum/6.2/performing-a-1d-unsteady-flow-analysis/calibration-of-unsteady-flow-models/roughness-coefficients), [FHWA HEC-20](https://www.fhwa.dot.gov/engineering/hydraulics/pubs/hif12004.pdf). The [USGS selection guide](https://www.usgs.gov/publications/guide-selecting-mannings-roughness-coefficients-natural-channels-and-flood-plains) supports assessing channel and floodplain characteristics rather than adopting a value from timing agreement alone.

**Interpretation used here:** 0.025–0.060 is a practical comparison band based on common natural-channel examples, not an official acceptance limit. Lower values can be credible for suitable alluvial beds; higher values need a roughness/vegetation explanation. This notebook does not establish those conditions at our sites. In particular, a low sand-bed base coefficient does not establish a whole-network effective coefficient.

The expanded **0.010–0.100** bounds are finite diagnostic search limits. We keep the best timing fit and the best tested fit within the common comparison band side by side. Fitted n remains conditional on BASED depth, slopes, width adjustments, segmentation and the chosen peak correspondence.

## 10. Refine the remaining lag discrepancies

Start from each initial best n and its neighbours. Recheck one shared clock for the expanded range. Re-evaluate the previous candidate on that same clock, then bisect sign changes in simulated minus observed lag. Stop once the residual is within half a forcing/output interval, or no unresolved crossing remains. This numerical stopping rule does not imply that observed peak timing is known that precisely.

The five previous boundary cases are allowed to explore beyond their original limits. Cases already sufficiently close need no further bisection. The original event windows, first-maximum definition and physical geometry remain fixed. Settlement, native flow checks and sparse-recurrence checks continue for every new trial.

In [ ]:
import travel_time_refinement as refinement
EXPANDED_BOUNDS = (.010, .100)
REFINEMENT_OUTPUT = ROOT / 'RAPID/outputs/travel_time_calibration_refined'
REFINEMENT_OUTPUT.mkdir(parents=True,exist_ok=True)
refined_cases, refinement_trials, refined_rows = {}, {}, []
for ex, previous_trials in trials_by_example.items():
    new_case = refinement.expanded_case(cases[ex],EXPANDED_BOUNDS)
    refined_cases[ex] = new_case
    trials, result = refinement.refine_case(new_case,previous_trials,REFINEMENT_OUTPUT)
    refinement_trials[ex] = trials
    refined_rows.append(result)
    pd.DataFrame(refined_rows).to_csv(REFINEMENT_OUTPUT/'refined_candidates.csv',index=False)
    print(f'{ex}: {result["previous_n"]:.5f} → {result["n_manning"]:.5f}; '
          f'error {result["previous_abs_error_h"]*60:.1f} → {result["abs_error_h"]*60:.1f} min; '
          f'{result["roughness_context"]}',flush=True)
refined_candidates = pd.DataFrame(refined_rows)
overview = refined_candidates[['example','previous_n','n_manning','previous_abs_error_h','abs_error_h',
                               'common_band_best_n','common_band_abs_error_h','roughness_context']].copy()
overview['previous_error_min'] = overview.pop('previous_abs_error_h')*60
overview['refined_error_min'] = overview.pop('abs_error_h')*60
overview['comparison_band_error_h'] = overview.pop('common_band_abs_error_h')
display(overview.round(5))

In [ ]:
ordered = refined_candidates.set_index('example').loc[list(refined_cases)]
fig,ax = plt.subplots(figsize=(12,4.5))
x=np.arange(len(ordered))
ax.scatter(x-.15,ordered.previous_abs_error_h,label='Initial best',color='#64748b',s=35)
ax.scatter(x,ordered.abs_error_h,label='Refined timing fit',color='#2563eb',s=35)
ax.scatter(x+.15,ordered.common_band_abs_error_h,label='Best tested within n=0.025–0.060',color='#d97706',marker='x',s=45)
ax.set(xticks=x,xticklabels=ordered.index,xlabel='Example',ylabel='Absolute peak-lag error (hours)',
       title='Refinement and the cost of restricting n to the common comparison band')
ax.set_yscale('symlog',linthresh=.1)
ax.set_ylim(bottom=0)
ax.legend(frameon=False,ncol=3,fontsize=9)
ax.grid(axis='y',alpha=.15)
fig.tight_layout()
fig.savefig(REFINEMENT_OUTPUT/'refinement_comparison.png',dpi=160)
plt.show()
print('Within observation sampling tolerance:',int(ordered.within_observation_sampling_tolerance.sum()),'of',len(ordered))
print('Remaining search-boundary candidates:',ordered.loc[ordered.best_at_search_boundary].index.tolist())

## 11. Travel-time fit versus absolute peak timing

Matching upstream-to-downstream **lag** does not remove a discrepancy between the applied forcing peak and the upstream gauge peak. The absolute outlet error is the sum of that forcing offset and the signed lag error. The table below keeps both quantities explicit, including any shift caused by processed forcing or competing maxima. Peak interpretation notes from the initial review continue to apply.

The physical-roughness interpretation remains provisional, including for values inside the comparison band. We have not classified site bed materials or floodplain vegetation here. Do not promote low/high coefficients solely because they give smaller timing residuals.

In [ ]:
display(refined_candidates[['example','n_manning','observed_lag_h','simulated_lag_h','signed_error_h',
                            'forcing_minus_observed_upstream_peak_h','outlet_minus_observed_downstream_peak_h',
                            'significant_forcing_peak_offset','target_notes']].round(4))
offset_identity = (refined_candidates.signed_error_h+
                   refined_candidates.forcing_minus_observed_upstream_peak_h-
                   refined_candidates.outlet_minus_observed_downstream_peak_h)
assert np.allclose(offset_identity,0,atol=1e-9)

In [ ]:
# Select a refined case. Case 46 also exposes the applied-forcing peak offset.
REFINED_PLOT_EXAMPLE = '46'
case = refined_cases[REFINED_PLOT_EXAMPLE]
chosen = refined_candidates.loc[refined_candidates.example.eq(REFINED_PLOT_EXAMPLE)].iloc[0]
_, original_wave = cal.run_trial(case,cal.BASE_N,REFINEMENT_OUTPUT)
_, refined_wave = cal.run_trial(case,float(chosen.n_manning),REFINEMENT_OUTPUT)
fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
traces=[
    (case['target']['series']['upstream'].time,case['target']['series']['upstream'].discharge_cms,'Observed upstream','#94a3b8'),
    (case['target']['series']['downstream'].time,case['target']['series']['downstream'].discharge_cms,'Observed downstream','#111827'),
    (original_wave.time_utc,original_wave.q_outlet_cms,'Baseline n=0.035','#64748b'),
    (refined_wave.time_utc,refined_wave.q_outlet_cms,f'Refined n={chosen.n_manning:.5f}','#2563eb'),
]
for times,values,label,color in traces:
    values=np.asarray(values)
    axes[0].plot(times,values,label=label,color=color,lw=1.3)
    axes[1].plot(times,(values-values.min())/np.ptp(values),color=color,lw=1.3)
axes[0].set(ylabel='Discharge (m³/s)',title=f'Example {REFINED_PLOT_EXAMPLE}: refined travel time, original UTC timestamps')
axes[0].legend(frameon=False,ncol=2)
axes[1].set(ylabel='Independent min–max scale',xlabel='Time (UTC)')
latest_peak=max(original_wave.time_utc.iloc[int(original_wave.q_outlet_cms.argmax())],
                refined_wave.time_utc.iloc[int(refined_wave.q_outlet_cms.argmax())])
axes[1].set_xlim(case['window'].first_sample-pd.Timedelta(hours=6),
                 max(case['window'].requested_end_exclusive,latest_peak+pd.Timedelta(days=1)))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
fig.tight_layout()
fig.savefig(REFINEMENT_OUTPUT/f'{REFINED_PLOT_EXAMPLE}_refined_hydrograph.png',dpi=160)
plt.show()
print(f'Lag residual: {chosen.signed_error_h:.3f} h; applied-forcing peak offset: '
      f'{chosen.forcing_minus_observed_upstream_peak_h:.3f} h; absolute outlet peak error: '
      f'{chosen.outlet_minus_observed_downstream_peak_h:.3f} h.')

## 12. Verify refined candidates and preserve results

Repeat the half-timestep check at the refined n values, keeping the forcing grid unchanged. Report any change in sampled peak time rather than assuming a timing fit is independent of the numerical clock. Cache identities include the refinement implementation and expanded-clock settings.

In [ ]:
refined_clock_rows=[]
for result in refined_rows:
    case=refined_cases[result['example']]
    fine=dict(case,routing_dt_s=case['routing_dt_s']/2,nsub=case['nsub']*2)
    fine['signature']=hashlib.sha256((case['signature']+':half-clock').encode()).hexdigest()
    check,_=cal.run_trial(fine,result['n_manning'],REFINEMENT_OUTPUT)
    change_s=3600*(check['simulated_lag_h']-result['simulated_lag_h'])
    refined_clock_rows.append(dict(example=result['example'],routing_dt_s=case['routing_dt_s'],
        half_dt_s=fine['routing_dt_s'],lag_change_seconds=change_s,
        within_one_forcing_interval=abs(change_s)<=case['forcing_step_s']+1e-8,
        minimum_native_flow_cms=check['minimum_native_flow_cms'],
        sparse_parity_max_cms=check['sparse_parity_max_cms']))
refined_clock_checks=pd.DataFrame(refined_clock_rows)
display(refined_clock_checks.round(6))
refined_clock_checks.to_csv(REFINEMENT_OUTPUT/'clock_checks.csv',index=False)
all_refinement_trials=pd.concat(refinement_trials.values(),ignore_index=True)
all_refinement_trials.to_csv(REFINEMENT_OUTPUT/'all_trials.csv',index=False)
summary=dict(examples=len(refined_candidates),search_trials=len(all_refinement_trials),
             median_initial_error_h=float(refined_candidates.previous_abs_error_h.median()),
             median_refined_error_h=float(refined_candidates.abs_error_h.median()),
             maximum_refined_error_h=float(refined_candidates.abs_error_h.max()),
             low_n_examples=refined_candidates.loc[refined_candidates.n_manning<.025,'example'].tolist(),
             high_n_examples=refined_candidates.loc[refined_candidates.n_manning>.060,'example'].tolist(),
             expanded_bounds=EXPANDED_BOUNDS,common_comparison_band=refinement.COMMON_BAND,
             workbook_sha256=schedule.attrs['workbook_sha256'],sources=refinement.SOURCES,
             literature_checked='2026-10-05',
             case_signatures={ex:c['signature'] for ex,c in refined_cases.items()},
             production_changed=False,site_roughness_verified=False,separate_events=False)
(REFINEMENT_OUTPUT/'refinement_manifest.json').write_text(json.dumps(summary,indent=2))
print(f'Refined {summary["examples"]} cases using {summary["search_trials"]} trial values. '
      f'Median residual: {summary["median_initial_error_h"]*60:.1f} → {summary["median_refined_error_h"]*60:.1f} min.')
print('Physical roughness remains unverified; candidates have not been applied to production.')